# English to Spanish with recurrent encoder and decoder

The bottleneck architecture, working — and its two limits, which are what the Transformer was invented to remove.

**Runs on:** GPU recommended — about 40 minutes on CPU &nbsp;·&nbsp; **Slides:** [Chapter 15 — Language Models and the Transformer](../../../course-web-slides/ch15/index.html) &nbsp;·&nbsp; **Section:** 02 — Sequence-to-sequence learning

---

## The data

In [ ]:
import pathlib, random, re, string
import keras
import tensorflow as tf

zip_path = keras.utils.get_file(
    origin=("http://storage.googleapis.com/download.tensorflow.org/"
            "data/spa-eng.zip"),
    fname="spa-eng", extract=True)
text_path = pathlib.Path(zip_path) / "spa-eng" / "spa.txt"

with open(text_path) as f:
    lines = f.read().split("\n")[:-1]

text_pairs = []
for line in lines:
    english, spanish = line.split("\t")
    spanish = "[start] " + spanish + " [end]"
    text_pairs.append((english, spanish))

print(f"{len(text_pairs):,} pairs")
print(random.choice(text_pairs))

`[start]` and `[end]` are inserted **in the data**, not built into the model. They are the seed and the stop signal for the generation loop.

## Two tokenizers, because punctuation is language-specific

In [ ]:
random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples:train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples:]

from keras import layers

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "").replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", "")

vocab_size, sequence_length = 15000, 20

english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size, output_mode="int",
    output_sequence_length=sequence_length)
spanish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size, output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization)

english_tokenizer.adapt([p[0] for p in train_pairs])
spanish_tokenizer.adapt([p[1] for p in train_pairs])
print("Spanish sequence length is one longer -- that extra slot is what")
print("makes the offset-by-one split possible.")

> ⚠️ **Two customisations, both easy to miss.** `[` and `]` must survive standardization or `"[start]"` collapses to `"start"`. And `¿` is not in `string.punctuation`, so it must be added explicitly.

## The pipeline

In [ ]:
batch_size = 64

def format_dataset(eng, spa):
    eng = english_tokenizer(eng)
    spa = spanish_tokenizer(spa)
    features = {"english": eng, "spanish": spa[:, :-1]}
    labels = spa[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    ds = tf.data.Dataset.from_tensor_slices((list(eng_texts), list(spa_texts)))
    ds = ds.batch(batch_size).map(format_dataset, num_parallel_calls=4)
    return ds.shuffle(2048).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

inputs, targets, weights = next(iter(train_ds))
for k, v in inputs.items():
    print(f"inputs[{k!r}]: {v.shape}")
print("targets:", targets.shape, " sample_weights:", weights.shape)

`sample_weights = labels != 0` tells Keras to **ignore padded positions** in the loss and metrics. Without it, a model that learned only to predict padding would score well.

## Why the naive single-RNN approach cannot work

> *"I will bring the bag to you"* becomes *"Te traeré la bolsa."* The **first** Spanish word corresponds to the **last** English word.

A single RNN emitting a target token at each step sees only source tokens 0…N when predicting target token N. There is no way to produce *Te* without having read to the end.

## Encoder and decoder

In [ ]:
embed_dim, hidden_dim = 256, 1024

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)
rnn_layer = layers.Bidirectional(layers.GRU(hidden_dim), merge_mode="sum")
encoder_output = rnn_layer(x)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)
x = layers.GRU(hidden_dim, return_sequences=True)(x, initial_state=encoder_output)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
seq2seq_rnn = keras.Model([source, target], target_predictions)
seq2seq_rnn.summary()

**Bidirectional in the encoder, emphatically not in the decoder.** We never predict source tokens, so there is nothing to cheat at — and a rich source representation is exactly what we want. The decoder is the Shakespeare setup from notebook 01, with its initial state supplied rather than zero.

## Training

In [ ]:
seq2seq_rnn.compile(optimizer="adam",
                    loss="sparse_categorical_crossentropy",
                    weighted_metrics=["accuracy"])
seq2seq_rnn.fit(train_ds, epochs=15, validation_data=val_ds, verbose=2)

About 65% next-token accuracy — and that metric is poor for translation. It assumes tokens 0…N are already correct when predicting N+1, which is exactly what is *not* true at inference. **BLEU** is the standard alternative.

## Generating translations

In [ ]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target = spanish_tokenizer([decoded_sentence])[:, :-1]
        preds = seq2seq_rnn.predict(
            [tokenized_input, tokenized_target], verbose=0)
        sampled_token_index = np.argmax(preds[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    s = random.choice(test_eng_texts)
    print("-")
    print(s)
    print(generate_translation(s))

> **Note** — This loop is **inefficient by construction**: it reprocesses the whole source and the whole generated target on every sampled word. Chapter 16 quantifies exactly how expensive that is, and what caching does about it.

## The two limits nothing here fixes

In [ ]:
print("1. THE BOTTLENECK")
print(f"   Everything the decoder knows about English arrives through")
print(f"   one vector of {hidden_dim} numbers. Longer or more complex")
print(f"   sentences do not get a bigger vector.\n")
print("2. FORGETTING")
print("   RNNs progressively lose the past. By the 100th token, little")
print("   remains of the start of the sequence.\n")
print("Deeper stacks, LSTM instead of GRU, a wider state -- none of these")
print("address either. Google Translate circa 2017 was seven large LSTM")
print("layers in essentially this shape, and these limits are what drove")
print("the search for something else.")

---

## What to take away

- An encoder compresses the whole source; a decoder generates from it token by token.
- Bidirectional in the encoder is right; in the decoder it destroys the objective.
- `sample_weights` keeps padding out of the loss.
- **The bottleneck and forgetting are structural** — no amount of tuning removes them.